# Drifting Diffusing Gaussian

This is a simple case to develop my own framework on top of FEniCSx. It solves the depth-averaged advection-diffusion equation on a square grid.

$$
\begin{cases}
    \frac{\mathrm{d} \iota}{\mathrm{d} t} + \nabla \cdot (\iota \mathbf{v} - \mathbf{D} \nabla c) = 0 &x \in \Omega \\
    \iota = \iota_in &x \in \Gamma_{\mathrm{in}} \\
    \mathbf{D} \nabla c \cdot \mathbf{\hat{n}} = 0 & x \in \Gamma_{\mathrm{out}}
\end{cases}
$$

## Imports and Setup

In [ ]:
import dolfinx as dx
import dolfinx.fem.petsc
import dolfinx.plot as dxp
import fenicsxtools as ft
from mpi4py import MPI
import numpy as np
from petsc4py import PETSc
import pyvista as pv
import ufl

In [ ]:
pv.set_jupyter_backend('static')
plotter = pv.Plotter()

## Problem Parameters

Here, we give more specifics for the problem. Specifically we are solving for a constant rightward flow and uniform diffusion in a square domain. The initial condition is a Gaussian bump, which should simply move rightwards and spread.

$$
\begin{align}
\mathbf{v} &= v_{0} \mathbf{\hat{e}}_{x} \\
\mathbf{D} &= D \mathbf{I} \\
\iota_{0}(\mathbf{x}) &= e^{-\frac{|\mathbf{x} - \mathbf{x}_{0}|^{2}}{2 \sigma^{2}}} \\
\iota(\mathbf{x}, t) &= \frac{\sigma^{2}}{\sigma^{2} + 2 d t} e^{-\frac{|\mathbf{x} - \mathbf{x}_{0} - \mathbf{v} t|^{2}}{2 (\sigma^{2} + 2 D t)}}
\end{align}
$$

In [ ]:
# Equation parameters
h0 = 1.0 # Constant height
u0 = 1.0 # Constant rightward flow
v0 = 0.0 # No vertical flow
d0 = 0.05 # Isotropic diffusion

In [ ]:
# Initial condition
x0 = 0.5
y0 = 0.5
r0 = 0.1
def iota0(x):
    return np.exp(-((x[0] - x0) ** 2 + (x[1] - y0) ** 2)/ (2 * r0 ** 2))

In [ ]:
# True solution
def iota_true(x, t):
    return r0 ** 2 / (r0 ** 2 + 2 * d0 * t) * np.exp(-((x[0] - x0) ** 2 + (x[1] - y0) ** 2)/ (2 * (r0 ** 2 + 2 * d0 * t)))

## Discretization and Meshing Parameters

A simple unit square $\Omega = (0, 1)^{2}$ is used for the domain.

In [ ]:
# Numerical parameters
nx = 10 # Cell resolution
dt = 0.001 # Time step
nt = 1000
write_every = 10 # Only every nth frame
plot_speedup = 1 # Relative to sim time

In [ ]:
# Define a simple
domain = dx.mesh.create_unit_square(MPI.COMM_WORLD, nx, nx, cell_type=dx.mesh.CellType.triangle)

## FEM Formulation

Here my library comes into use.
- The user defines function spaces and relevant functions.
- A template for depth-averaged advection-diffusion assembles them into the correct PDE.
- This is then fed through a formulation, which constructs the appropriate semidiscrete equations.
- The semidiscrete equations are then solved, either implicitly or explicitly.

In [ ]:
# Declare function spaces
# For now I'm using piecewise linear basis functions
V = dx.fem.functionspace(domain, ('Discontinuous Lagrange', 1))
W = dx.fem.functionspace(domain, ('Discontinuous Lagrange', 1, (domain.geometry.dim,)))

In [ ]:
# Initialize functions (solution variables and field variables)
# The for
iota = dx.fem.Function(V)

In [ ]:
# Generate strong form
# A depth-averaged advection-diffusion template is used
equation = ft.equations.get_depth_averaged_advection_diffusion(
    domain=domain,
    iota=None,
    h=None,
    v=None,
    D=None
)

In [ ]:
# Generate weak form
# A LDG formulation is used
# An alternating or downwind flux is generally used for the gradiend
formulation = ft.formulations.LDGFormulation(
    equation=equation,
    flux_treatment=ft.fluxn_llf_scalar,
    auxiliary_treatment=ft.fluxn_llf_downwind_scalar
)

In [ ]:
# Generate semi-discrete equations

## Solution Loop

In [ ]:
# Solution loop